# CVAE ablation — overfit / underfit diagnostics

Two sanity-check experiments on the CVAE ablation (`CVAEAutoEncoder`). Neither is a paper experiment; both exist to debug the training pipeline before committing to a long run.

**Test A — Overfit a tiny subset.** Take ~32 examples and the full-size model. With a high LR and enough epochs, the loss should plummet toward zero. If it doesn't, the model lacks capacity *or* the training loop is wired wrong (bad gradient flow, frozen params, broken loss).

**Test B — Starve the model on the full dataset.** Shrink the CVAE bottleneck (`z_dim`, decoder hidden) and train normally. Train loss should *plateau high*. We also carve a val split off the train set so we can read the generalization gap. A tight gap with high loss is textbook underfit; a wide gap is the model still memorizing what little it can fit.

Together these bracket the model's behavior: the overfit run says "the loss can be minimized," the underfit run says "the loss is a meaningful signal when capacity is removed."

## 1. Configuration

In [ ]:
SCENE = "eth"           # any one scene's _train.pkl is fine for these tests
SEED = 123
DEVICE_PREF = "auto"    # auto-picks cuda > mps > cpu

# ── Overfit test config ──
OVERFIT_N = 32          # how many training examples to keep
OVERFIT_EPOCHS = 200
OVERFIT_LR = 5e-3       # higher LR — we want fast memorization
OVERFIT_BATCH = 32      # full subset per batch

# Full-size CVAE — the overfit model
ENCODER_DIM = 256
OVERFIT_Z_DIM = 32
OVERFIT_DECODER_HIDDEN = 256
OVERFIT_FUTURE_HIDDEN = 128

# ── Underfit test config ──
UNDERFIT_EPOCHS = 20
UNDERFIT_LR = 1e-3
UNDERFIT_BATCH = 64
UNDERFIT_Z_DIM = 2           # tiny latent — main bottleneck
UNDERFIT_DECODER_HIDDEN = 16
UNDERFIT_FUTURE_HIDDEN = 16
UNDERFIT_VAL_FRACTION = 0.1  # carve val off the train set for generalization-gap tracking

## 2. Setup: paths, device, seeds

Same `PROJECT_ROOT` and device-picking logic as `train.ipynb` / `eval.ipynb`. The seeds make the subset selection and the random_split reproducible.

In [ ]:
import os
import sys
import random
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Subset, DataLoader, random_split
from tqdm.auto import tqdm

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from mid_model import (
    load_environment, build_dataloader, get_hyperparameters,
    CVAEAutoEncoder,
)
# `collate` lives in dataset.preprocessing — it's not re-exported from mid_model.dataset.
from dataset.preprocessing import collate
from models.trajectron import Trajectron
from utils.model_registrar import ModelRegistrar

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 3. Shared data loading

Both tests use the same underlying `Environment` — they differ only in subsetting and in which model they wrap around. We load once, build the standard PEDESTRIAN dataloader with augmentation off (the `Subset` wrapper used below loses access to the `.augment` flag), and reuse the resulting `NodeTypeDataset` for both tests.

In [ ]:
pkl_path = os.path.join(PROJECT_ROOT, "processed_data", f"{SCENE}_train.pkl")
env = load_environment(pkl_path)
print(f"Loaded {pkl_path}")
print(f"  scenes: {len(env.scenes)}")
print(f"  total nodes: {sum(len(s.nodes) for s in env.scenes)}")

hyperparams = get_hyperparameters(encoder_dim=ENCODER_DIM)

# build_dataloader returns the DataLoader + node_type. We pull the
# underlying NodeTypeDataset off the loader and rebuild fresh loaders
# below — once for the Subset (overfit), once for the random_split
# (underfit train/val).
base_loader, node_type = build_dataloader(
    env=env,
    hyperparams=hyperparams,
    batch_size=OVERFIT_BATCH,
    shuffle=True,
    num_workers=0,
    augment=False,
)
base_dataset = base_loader.dataset
print(f"\nNode type: {node_type}")
print(f"Full NodeTypeDataset size: {len(base_dataset)}")

## Section A — Overfit test

Take a tiny random subset (32 examples) and train the full-size CVAE with a high LR. If the implementation is correct, the model should memorize these examples and the loss should collapse toward zero. A failure mode here usually means:

- gradients aren't flowing (some module frozen by accident)
- the loss is computed against the wrong target
- the model is genuinely too small (unlikely with full-size CVAE on 32 examples)

In [ ]:
# Wrap the NodeTypeDataset in a Subset of the first OVERFIT_N indices.
# We then build a fresh DataLoader around the Subset, reusing the same
# `collate` that EnvironmentDataset's loader uses (otherwise the batch
# 9-tuple structure breaks).
subset_indices = list(range(min(OVERFIT_N, len(base_dataset))))
overfit_subset = Subset(base_dataset, subset_indices)

overfit_loader = DataLoader(
    overfit_subset,
    collate_fn=collate,
    batch_size=OVERFIT_BATCH,
    shuffle=True,
    num_workers=0,
)
print(f"Overfit subset size: {len(overfit_subset)}")
print(f"Overfit batches per epoch: {len(overfit_loader)}")

In [ ]:
# Fresh encoder + registrar so we don't carry any state between the two
# tests. Same construction sequence as train.ipynb: registrar → Trajectron
# → set_environment → set_annealing_params → CVAEAutoEncoder.
overfit_registrar = ModelRegistrar(model_dir="../checkpoints", device=DEVICE)
overfit_encoder = Trajectron(overfit_registrar, hyperparams, DEVICE)
overfit_encoder.set_environment(env)
overfit_encoder.set_annealing_params()

overfit_model = CVAEAutoEncoder(
    encoder=overfit_encoder,
    registrar=overfit_registrar,
    encoder_dim=ENCODER_DIM,
    z_dim=OVERFIT_Z_DIM,
    decoder_hidden=OVERFIT_DECODER_HIDDEN,
    future_hidden=OVERFIT_FUTURE_HIDDEN,
    kl_weight=1.0,
).to(DEVICE)

overfit_param_count = sum(p.numel() for p in overfit_model.parameters())
print(f"Overfit (full-size) model params: {overfit_param_count:,}")

In [ ]:
overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=OVERFIT_LR)
overfit_losses = []

overfit_model.train()
pbar = tqdm(range(1, OVERFIT_EPOCHS + 1), desc="Overfit", ncols=100)
for epoch in pbar:
    epoch_losses = []
    for batch in overfit_loader:
        overfit_optimizer.zero_grad()
        loss = overfit_model.get_loss(batch, node_type)
        loss.backward()
        overfit_optimizer.step()
        epoch_losses.append(loss.item())
    avg = float(np.mean(epoch_losses))
    overfit_losses.append(avg)
    pbar.set_postfix(loss=f"{avg:.4f}")
    if epoch == 1 or epoch % 20 == 0:
        print(f"  epoch {epoch:>3}: avg loss = {avg:.4f}")

In [ ]:
# Soft sanity check + plot. We're looking for an obvious downward trend;
# the exact final value depends on the KL term (which has a floor > 0).
init_loss = overfit_losses[0]
final_loss = overfit_losses[-1]
ratio = final_loss / max(init_loss, 1e-12)
print(f"Initial loss: {init_loss:.4f}")
print(f"Final loss:   {final_loss:.4f}")
print(f"Final/initial ratio: {ratio:.3f}")

if ratio > 0.2:
    print("  WARNING: final loss > 20% of initial — model may not have capacity")
    print("           to overfit, or LR too low, or gradients not flowing.")
else:
    print("  OK: loss dropped to <=20% of initial — training loop looks wired correctly.")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(overfit_losses) + 1), overfit_losses, color="steelblue")
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel("avg loss (log)")
ax.set_title(f"Overfit test: {OVERFIT_N} examples, full-size CVAE")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## Section B — Underfit test

Train a deliberately starved CVAE (tiny `z_dim`, tiny decoder hidden) on the full dataset. Expectation: the loss plateaus high and the train/val gap is small — the model is too small to memorize, so any error it shows is bias, not variance. If the val curve diverges upward despite the bottleneck, something is letting the model overfit anyway (KL too weak, data too small to be representative, etc.).

In [ ]:
# Carve a val split off the same NodeTypeDataset we built above.
# `random_split` doesn't preserve attributes like .node_type, so we keep a
# reference to `node_type` from the original loader. The collate_fn is
# what restores the batch 9-tuple shape that the encoder expects.
n_total = len(base_dataset)
n_val = int(round(UNDERFIT_VAL_FRACTION * n_total))
n_train = n_total - n_val
split_gen = torch.Generator().manual_seed(SEED)
underfit_train_subset, underfit_val_subset = random_split(
    base_dataset, [n_train, n_val], generator=split_gen
)

underfit_train_loader = DataLoader(
    underfit_train_subset,
    collate_fn=collate,
    batch_size=UNDERFIT_BATCH,
    shuffle=True,
    num_workers=0,
)
underfit_val_loader = DataLoader(
    underfit_val_subset,
    collate_fn=collate,
    batch_size=UNDERFIT_BATCH,
    shuffle=False,
    num_workers=0,
)
print(f"Underfit train: {len(underfit_train_subset)}  ({len(underfit_train_loader)} batches)")
print(f"Underfit val:   {len(underfit_val_subset)}  ({len(underfit_val_loader)} batches)")

In [ ]:
# Fresh encoder + registrar again. The encoder side is unchanged from the
# overfit model — what's starved is the CVAE decoder (z_dim, hidden dims).
underfit_registrar = ModelRegistrar(model_dir="../checkpoints", device=DEVICE)
underfit_encoder = Trajectron(underfit_registrar, hyperparams, DEVICE)
underfit_encoder.set_environment(env)
underfit_encoder.set_annealing_params()

underfit_model = CVAEAutoEncoder(
    encoder=underfit_encoder,
    registrar=underfit_registrar,
    encoder_dim=ENCODER_DIM,
    z_dim=UNDERFIT_Z_DIM,
    decoder_hidden=UNDERFIT_DECODER_HIDDEN,
    future_hidden=UNDERFIT_FUTURE_HIDDEN,
    kl_weight=1.0,
).to(DEVICE)

underfit_param_count = sum(p.numel() for p in underfit_model.parameters())
print(f"Underfit (starved) model params: {underfit_param_count:,}")
print(f"Overfit  (full-size) model params: {overfit_param_count:,}")
print(f"Ratio (starved / full): {underfit_param_count / overfit_param_count:.3f}")

In [ ]:
underfit_optimizer = torch.optim.Adam(underfit_model.parameters(), lr=UNDERFIT_LR)
underfit_train_losses = []
underfit_val_losses = []

for epoch in range(1, UNDERFIT_EPOCHS + 1):
    # Train pass
    underfit_model.train()
    train_losses_epoch = []
    pbar = tqdm(underfit_train_loader, desc=f"Underfit epoch {epoch}/{UNDERFIT_EPOCHS}", ncols=100)
    for batch in pbar:
        underfit_optimizer.zero_grad()
        loss = underfit_model.get_loss(batch, node_type)
        loss.backward()
        underfit_optimizer.step()
        train_losses_epoch.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    avg_train = float(np.mean(train_losses_epoch))
    underfit_train_losses.append(avg_train)

    # Val pass
    underfit_model.eval()
    val_losses_epoch = []
    with torch.no_grad():
        for batch in underfit_val_loader:
            val_loss = underfit_model.get_loss(batch, node_type)
            val_losses_epoch.append(val_loss.item())
    avg_val = float(np.mean(val_losses_epoch))
    underfit_val_losses.append(avg_val)
    underfit_model.train()

    print(f"  epoch {epoch:>2}: train = {avg_train:.4f}  val = {avg_val:.4f}  gap = {avg_val - avg_train:+.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
xs = range(1, len(underfit_train_losses) + 1)
ax.plot(xs, underfit_train_losses, color="steelblue", label="train")
ax.plot(xs, underfit_val_losses, color="darkorange", label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("avg loss")
ax.set_title(f"Underfit test: starved CVAE (z_dim={UNDERFIT_Z_DIM}, hidden={UNDERFIT_DECODER_HIDDEN})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

final_gap = underfit_val_losses[-1] - underfit_train_losses[-1]
print(f"Final train loss: {underfit_train_losses[-1]:.4f}")
print(f"Final val loss:   {underfit_val_losses[-1]:.4f}")
print(f"Final gap:        {final_gap:+.4f}")
if abs(final_gap) < 0.05 * underfit_train_losses[-1]:
    print("  Interpretation: train and val track closely — clean underfit (high bias, low variance).")
elif final_gap > 0:
    print("  Interpretation: val above train — model is still overfitting despite the bottleneck.")
    print("                  Look at KL weight or dataset size before reading this as a real underfit.")
else:
    print("  Interpretation: val below train — likely noise from a small val set, not a real signal.")

## Side-by-side comparison

Both curves on one figure. Left panel: overfit run (log y). Right panel: underfit train + val. The overfit curve should fall off a cliff; the underfit curves should flatten out well above the floor.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, len(overfit_losses) + 1), overfit_losses, color="steelblue")
axes[0].set_yscale("log")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("avg loss (log)")
axes[0].set_title(f"Overfit: {OVERFIT_N} examples, full model")
axes[0].grid(True, which="both", alpha=0.3)

xs = range(1, len(underfit_train_losses) + 1)
axes[1].plot(xs, underfit_train_losses, color="steelblue", label="train")
axes[1].plot(xs, underfit_val_losses, color="darkorange", label="val")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("avg loss")
axes[1].set_title(f"Underfit: full data, starved model (z={UNDERFIT_Z_DIM})")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Interpretation guide

- **Overfit succeeds (loss → near zero).** The training loop is correctly wired: encoder gradients flow, the loss targets the right tensor, the optimizer covers all parameters. This is a precondition for trusting any longer training run.
- **Overfit fails (loss stays flat).** Something is wrong before the data even matters: some submodule is frozen, the loss is computed against the wrong target, or the LR is far too low. Investigate before scaling up.
- **Underfit shows a small train/val gap with high loss.** High bias, low variance — the starved model genuinely can't represent the data. The loss is a meaningful signal of model capacity.
- **Underfit shows a large val − train gap.** The model is still memorizing despite the bottleneck. Usually this means the KL term is too weak to regularize a small `z_dim`, or the training set is so small that even a tiny model can index it. Worth checking before reading later runs as "underfit."

This notebook is debugging infrastructure, not a result. Run it once after wiring changes to the model or training loop; don't include it in any paper-style sweep.